# Ngày 17 · Gradient boosting cho hồi quy

- Viết vòng boosting: mỗi cây nhỏ khớp phần dư y − F
- Thấy vai trò của learning rate và số cây
- Chọn số cây bằng sai số validation theo từng giai đoạn (early stopping)

**Đọc:**
- ISLR (bản 2), §8.2.3 Boosting (Algorithm 8.2)
- Friedman (2001), Greedy Function Approximation: A Gradient Boosting Machine, Annals of Statistics 29(5), 1189–1232
- Hastie, Tibshirani & Friedman, ESL, §10.10 Numerical Optimization via Gradient Boosting

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
# Diabetes (Efron et al., 2004), có sẵn trong scikit-learn: 442 bệnh nhân, 10 đặc trưng, y là chỉ số tiến triển sau 1 năm.
X_db, y_db = load_diabetes(return_X_y=True)
Xd_tr, Xd_va, yd_tr, yd_va = train_test_split(X_db, y_db, test_size=0.3, random_state=0)
_sk17 = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=0).fit(Xd_tr, yd_tr)
def _ref_staged(model, X, y):
    F, out = np.full(len(X), model['f0']), []
    for tree in model['trees']:
        F = F + model['lr'] * tree.predict(X)
        out.append(float(np.mean((y - F) ** 2)))
    return np.array(out)


## Boosting từ cây hồi quy và early stopping

Gợi ý: F khởi đầu bằng mean(y). Mỗi vòng: r = y − F, fit DecisionTreeRegressor(max_depth=max_depth, random_state=0) lên (X, r), rồi F += lr · tree.predict(X). Lưu f0, lr và danh sách cây vào dict.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

def fit_gbm(X, y, n_estimators, lr, max_depth):
    # Trả {'f0': float, 'lr': lr, 'trees': [cây 1, ..., cây n]}.
    raise NotImplementedError

def predict_gbm(model, X):
    raise NotImplementedError

def staged_mse(model, X, y):
    # Mảng dài n_estimators: MSE sau khi cộng cây thứ 1, 2, ...
    raise NotImplementedError

def best_n_estimators(model, X_val, y_val):
    # Số cây (đếm từ 1) cho MSE validation nhỏ nhất.
    raise NotImplementedError


In [ ]:
_c17 = {}
def _m17():
    if 'm' not in _c17:
        _c17['m'] = fit_gbm(Xd_tr, yd_tr, 100, 0.1, 2)
    return _c17['m']
_report('d17-boost',
    _check('Dự đoán trên train khớp GradientBoostingRegressor', lambda: (np.allclose(predict_gbm(_m17(), Xd_tr), _sk17.predict(Xd_tr), atol=1e-6), '100 cây, lr = 0,1, max_depth = 2, loss bình phương.')),
    _check('staged_mse trên train khớp staged_predict và không tăng', lambda: ((lambda s: np.allclose(s, [np.mean((yd_tr - p) ** 2) for p in _sk17.staged_predict(Xd_tr)]) and np.all(np.diff(s) <= 1e-9))(staged_mse(_m17(), Xd_tr, yd_tr)), 'Lỗi train của boosting giảm đơn điệu.')),
    _check('best_n_estimators chọn đúng số cây theo validation', lambda: (best_n_estimators(_m17(), Xd_va, yd_va) == int(np.argmin(_ref_staged(_m17(), Xd_va, yd_va)) + 1), 'Lỗi validation giảm rồi tăng: đó là chỗ dừng sớm.')),
)


## Dự án

Trên House Prices với log(SalePrice), so RandomForestRegressor với HistGradientBoostingRegressor(early_stopping=True, validation_fraction=0.1, random_state=0) theo RMSE 5-fold. Thử learning_rate ∈ {0,3; 0,1; 0,03} và ghi n_iter_ mà early stopping chọn. Xuất submission và tự nộp trên Kaggle (nhớ đổi ngược bằng np.expm1 nếu bạn dùng log1p).